In [2]:
# -*- coding: utf-8 -*-
"""
BATCH PROCESSING FOR SARCASM DETECTION
This script performs sarcasm detection on hotel reviews using batch processing.
"""

# =============================================================================
# INSTALLATION AND IMPORTS
# =============================================================================
print("Installing required packages...")
!pip install pandas numpy
!pip install emot langdetect

import pandas as pd
import numpy as np
import re
import time
import json
import warnings
import requests
from typing import List, Dict, Any
warnings.filterwarnings('ignore')

# Import for preprocessing
from langdetect import detect, DetectorFactory, LangDetectException
from langdetect import detect_langs
from emot.emo_unicode import EMOTICONS_EMO

# Set random seeds for reproducibility
DetectorFactory.seed = 0
np.random.seed(42)

# =============================================================================
# CONFIGURATION
# =============================================================================
# File paths (adjust based on your Google Drive structure)
INPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/hotels.xlsx"
OUTPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/hotels_sarcasm_batch_processed.xlsx"

# API settings (using a free API as fallback)
API_URL = "https://api-inference.huggingface.co/models/cardiffnlp/twitter-roberta-base-sarcasm"
API_TOKEN = "your_huggingface_token_here"  # Replace with your token

# Processing settings
BATCH_SIZE = 5  # Smaller batch size for API limits
DELAY_BETWEEN_BATCHES = 2  # Delay between batch requests in seconds
REVIEW_COLUMN = "Review Summary"  # Column containing review text
SAMPLE_SIZE = 200  # Set to None for full dataset, or number for testing

# =============================================================================
# GOOGLE DRIVE SETUP
# =============================================================================
print("Mounting Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

# =============================================================================
# PREPROCESSING FUNCTIONS
# =============================================================================
def handle_emoticons(text):
    """
    Convert emoticons to text descriptions.
    Important for sarcasm analysis.
    Example: :) → "smiling face"
    """
    if not hasattr(handle_emoticons, 'emoticon_patterns'):
        handle_emoticons.emoticon_patterns = []
        for emoticon, description in EMOTICONS_EMO.items():
            escaped_emoticon = re.escape(emoticon)
            handle_emoticons.emoticon_patterns.append(
                (re.compile(escaped_emoticon), f" {description} ")
            )

    text = str(text)
    for pattern, replacement in handle_emoticons.emoticon_patterns:
        text = pattern.sub(replacement, text)
    return text

def is_english_reliable(text):
    """
    Detect if text is English with high confidence.
    Filters out non-English reviews to improve analysis quality.
    """
    if pd.isna(text) or text == "" or len(str(text).strip()) < 10:
        return False

    text = str(text)

    try:
        text_sample = text[:500].strip()
        lang_probabilities = detect_langs(text_sample)

        for lang_prob in lang_probabilities:
            if lang_prob.lang == 'en' and lang_prob.prob >= 0.7:
                return True
            elif lang_prob.lang == 'en' and lang_prob.prob >= 0.5 and len(text_sample) > 50:
                return True

        return False

    except (LangDetectException, Exception):
        text_lower = text.lower()
        english_words = {'the', 'and', 'to', 'of', 'a', 'in', 'that', 'is', 'it', 'for', 'was', 'but', 'with', 'on', 'he', 'she', 'they', 'we', 'you', 'i'}
        words = re.findall(r'\b[a-z]+\b', text_lower)

        if not words or len(words) < 5:
            return False

        english_count = sum(1 for word in words if word in english_words)
        return (english_count / len(words)) > 0.3

def preprocess_for_sarcasm(text):
    """
    Specialized preprocessing for sarcasm detection.
    Preserves case, punctuation, and linguistic cues important for sarcasm.
    """
    if pd.isna(text) or text == "":
        return ""

    text = str(text)

    # Basic cleaning only - preserve case and punctuation!
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\s+', ' ', text)

    # Handle emoticons (crucial for sarcasm)
    text = handle_emoticons(text)

    return text.strip()

# =============================================================================
# BATCH SARCASM DETECTION MODULE
# =============================================================================
class BatchSarcasmDetector:
    """API-based sarcasm detection using batch processing"""

    def __init__(self, batch_size=BATCH_SIZE):
        self.batch_size = batch_size
        self.headers = {"Authorization": f"Bearer {API_TOKEN}"} if API_TOKEN != "your_huggingface_token_here" else {}

    def query_api(self, payload):
        """Query the HuggingFace API for sarcasm detection"""
        try:
            response = requests.post(API_URL, headers=self.headers, json=payload)
            return response.json()
        except Exception as e:
            print(f"API request failed: {e}")
            return None

    def analyze_batch(self, reviews_batch: List[str]) -> List[Dict[str, Any]]:
        """Analyze a batch of reviews using the API"""
        if not reviews_batch:
            return []

        # Prepare payload for the API
        payload = {"inputs": reviews_batch}

        try:
            response = self.query_api(payload)

            if response and isinstance(response, list):
                results = []
                for item in response:
                    if isinstance(item, list) and len(item) > 0:
                        # Get the first (and usually only) result for each review
                        result = item[0]
                        sarcasm_score = result.get('score', 0) if result.get('label', '') == 'sarcasm' else 0

                        results.append({
                            'sarcasm': sarcasm_score > 0.5,
                            'confidence': sarcasm_score,
                            'reason': 'API-based detection',
                            'type': 'positive_sarcasm' if sarcasm_score > 0.5 else 'none'
                        })
                    else:
                        # Fallback if response format is unexpected
                        results.append(self.heuristic_fallback(""))

                return results
            else:
                # Fallback if API fails
                return [self.heuristic_fallback(review) for review in reviews_batch]

        except Exception as e:
            print(f"Error processing batch: {e}")
            return [self.heuristic_fallback(review) for review in reviews_batch]

    def heuristic_fallback(self, text):
        """Fallback method when API analysis fails"""
        text_lower = text.lower() if text else ""
        sarcastic_keywords = [
            'sarcastic', 'ironic', 'obviously', 'of course', 'surprise',
            'wow', 'really', 'just what i needed', 'thanks for', 'perfect',
            'as expected', 'big surprise', 'shockingly', 'unbelievably'
        ]

        # More sophisticated heuristic checks
        is_sarcastic = any(kw in text_lower for kw in sarcastic_keywords)

        # Check for exaggerated praise in short reviews
        if not is_sarcastic and text_lower:
            positive_words = ['excellent', 'amazing', 'perfect', 'fantastic', 'outstanding', 'wonderful']
            has_exaggeration = sum(1 for word in positive_words if word in text_lower) >= 2
            is_short = len(text_lower.split()) < 10
            is_sarcastic = has_exaggeration and is_short

        # Check for negative words in positive context
        if not is_sarcastic and text_lower:
            negative_words = ['terrible', 'awful', 'horrible', 'disappointing', 'bad']
            has_negative = any(word in text_lower for word in negative_words)
            has_positive = any(word in text_lower for word in positive_words)
            is_sarcastic = has_negative and has_positive

        confidence = 0.7 if is_sarcastic else 0.3

        return {
            'sarcasm': is_sarcastic,
            'confidence': confidence,
            'reason': 'Heuristic fallback detection',
            'type': 'positive_sarcasm' if is_sarcastic else 'none'
        }

    def process_dataframe(self, df, review_column=REVIEW_COLUMN):
        """Process entire dataframe using batch processing"""
        print(f"Processing {len(df)} reviews for sarcasm using batch size {self.batch_size}...")

        df_sarcasm = df.copy()
        df_sarcasm['sarcasm'] = 0  # 0 = not sarcastic, 1 = sarcastic
        df_sarcasm['sarcasm_confidence'] = 0.0
        df_sarcasm['sarcasm_reason'] = 'Not processed'
        df_sarcasm['sarcasm_type'] = 'none'
        df_sarcasm['processed_review'] = df_sarcasm[review_column].apply(preprocess_for_sarcasm)

        # Filter out empty reviews
        valid_indices = []
        valid_reviews = []

        for idx, row in df_sarcasm.iterrows():
            if pd.isna(row['processed_review']) or row['processed_review'].strip() == '':
                # Apply fallback to empty reviews immediately
                result = self.heuristic_fallback("")
                df_sarcasm.at[idx, 'sarcasm'] = 1 if result.get('sarcasm', False) else 0
                df_sarcasm.at[idx, 'sarcasm_confidence'] = result.get('confidence', 0.0)
                df_sarcasm.at[idx, 'sarcasm_reason'] = str(result.get('reason', ''))[:200]
                df_sarcasm.at[idx, 'sarcasm_type'] = result.get('type', 'none')
            else:
                valid_indices.append(idx)
                valid_reviews.append(row['processed_review'])

        # Process valid reviews in batches
        total_batches = (len(valid_reviews) + self.batch_size - 1) // self.batch_size
        batch_count = 0

        for i in range(0, len(valid_reviews), self.batch_size):
            batch_count += 1
            batch_indices = valid_indices[i:i+self.batch_size]
            batch_reviews = valid_reviews[i:i+self.batch_size]

            print(f"Processing batch {batch_count}/{total_batches} ({len(batch_reviews)} reviews)")

            try:
                batch_results = self.analyze_batch(batch_reviews)

                # Apply results to dataframe
                for idx, result in zip(batch_indices, batch_results):
                    df_sarcasm.at[idx, 'sarcasm'] = 1 if result.get('sarcasm', False) else 0
                    df_sarcasm.at[idx, 'sarcasm_confidence'] = result.get('confidence', 0.0)
                    df_sarcasm.at[idx, 'sarcasm_reason'] = str(result.get('reason', ''))[:200]
                    df_sarcasm.at[idx, 'sarcasm_type'] = result.get('type', 'none')

            except Exception as e:
                print(f"Error processing batch {batch_count}: {e}")
                # Apply fallback to all reviews in this batch
                for idx in batch_indices:
                    result = self.heuristic_fallback(df_sarcasm.at[idx, 'processed_review'])
                    df_sarcasm.at[idx, 'sarcasm'] = 1 if result.get('sarcasm', False) else 0
                    df_sarcasm.at[idx, 'sarcasm_confidence'] = result.get('confidence', 0.0)
                    df_sarcasm.at[idx, 'sarcasm_reason'] = str(result.get('reason', ''))[:200]
                    df_sarcasm.at[idx, 'sarcasm_type'] = result.get('type', 'none')

            # Delay between batches to avoid overwhelming the API
            if batch_count < total_batches:
                time.sleep(DELAY_BETWEEN_BATCHES)

        sarcastic_count = df_sarcasm['sarcasm'].sum()
        print(f"Sarcasm detection completed! Sarcastic reviews: {sarcastic_count}/{len(df_sarcasm)}")

        return df_sarcasm

# =============================================================================
# MAIN EXECUTION
# =============================================================================
def main():
    """
    Main function that runs batch sarcasm detection on hotel reviews.
    """
    print("="*60)
    print("BATCH SARCASM DETECTION PIPELINE")
    print("="*60)

    # Step 1: Load data
    print("1. Loading data...")
    try:
        df = pd.read_excel(INPUT_FILE)
        print(f"   Loaded {len(df)} reviews from {INPUT_FILE}")
    except Exception as e:
        print(f"   Error loading file: {e}")
        return None

    # For testing, use a subset
    if SAMPLE_SIZE and SAMPLE_SIZE < len(df):
        print(f"   Using sample of {SAMPLE_SIZE} reviews for testing")
        df = df.head(SAMPLE_SIZE).copy()

    # Step 2: Filter English reviews
    print("2. Filtering English reviews...")
    initial_count = len(df)
    english_mask = df[REVIEW_COLUMN].apply(is_english_reliable)
    df = df[english_mask]
    print(f"   Removed {initial_count - len(df)} non-English reviews")

    if len(df) == 0:
        print("   No English reviews found!")
        return None

    # Step 3: Run batch sarcasm detection
    print("3. Running batch sarcasm detection...")
    detector = BatchSarcasmDetector(batch_size=BATCH_SIZE)
    df_result = detector.process_dataframe(df)

    # Step 4: Save results
    print("4. Saving results...")
    df_result.to_excel(OUTPUT_FILE, index=False)
    print(f"   Results saved to {OUTPUT_FILE}")

    # Step 5: Display summary
    print("\n" + "="*60)
    print("ANALYSIS SUMMARY")
    print("="*60)
    print(f"Total reviews analyzed: {len(df_result)}")

    if 'sarcasm' in df_result.columns:
        sarcastic = df_result['sarcasm'].sum()
        print(f"Sarcastic reviews detected: {sarcastic} ({sarcastic/len(df_result)*100:.1f}%)")

        # Show some examples
        print("\nSample of sarcastic reviews:")
        sarcastic_samples = df_result[df_result['sarcasm'] == 1].head(3)
        for _, row in sarcastic_samples.iterrows():
            print(f"  - {row[REVIEW_COLUMN][:100]}... (Confidence: {row['sarcasm_confidence']:.2f})")

    return df_result

# =============================================================================
# EXECUTION
# =============================================================================
if __name__ == "__main__":
    print("Starting batch sarcasm detection pipeline...")
    results = main()

    if results is not None:
        print("\n✅ Analysis completed successfully!")
        print(f"Final results contain {len(results)} reviews")
    else:
        print("\n❌ Analysis failed!")

Installing required packages...
Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Starting batch sarcasm detection pipeline...
BATCH SARCASM DETECTION PIPELINE
1. Loading data...
   Loaded 6780 reviews from /content/drive/MyDrive/Colab Notebooks/hotels.xlsx
   Using sample of 200 reviews for testing
2. Filtering English reviews...
   Removed 80 non-English reviews
3. Running batch sarcasm detection...
Processing 120 reviews for sarcasm using batch size 5...
Processing batch 1/24 (5 reviews)
Processing batch 2/24 (5 reviews)
Processing batch 3/24 (5 reviews)
Processing batch 4/24 (5 reviews)
Processing batch 5/24 (5 reviews)
Processing batch 6/24 (5 reviews)
Processing batch 7/24 (5 reviews)
Processing batch 8/24 (5 reviews)
Processing batch 9/24 (5 reviews)
Processing batch 10/24 (5 reviews)
Processing batch 11/24 (5 reviews)
Processing batch 12/24 (5 reviews)
Processing batch 13/24 